In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *
from biked_commons.conditioning import conditioning

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmark_models\generative_models\../../..\biked_commons\prediction\usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malici

In [2]:
def sample_continuous(num_samples, split="test", randomize = False):
    emb = conditioning.sample_image_embedding(num_samples, split, randomize)
    rider = conditioning.sample_riders(num_samples, split, randomize)
    use_case = conditioning.sample_use_case(num_samples, split, randomize)
    all = torch.cat((emb, rider, use_case), dim=1)
    return all

def parse_continuous_condition(condition):
    image_embeddings = condition[:, :512]
    use_case_condition = condition[:, -3:]
    rider_condition = condition[:, 512:-3]
    condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
    return condition


In [3]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)


In [4]:
cond = parse_continuous_condition(sample_continuous(len(data), randomize=True)[0].unsqueeze(0))


evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)



ref_point = get_ref_point(evaluator, requirement_names)


In [5]:
requirement_names

['Usability Score - 0 to 1',
 'Drag Force',
 'Knee Angle Error',
 'Hip Angle Error',
 'Arm Angle Error',
 'Mass',
 'Planar Compliance',
 'Transverse Compliance',
 'Eccentric Compliance',
 'Planar Safety Factor',
 'Eccentric Safety Factor',
 'Saddle height too small',
 'Seat post too short',
 'Head tube lower extension too great',
 'Head tube length too great',
 'Certain parameters must be positive',
 'Chain stay should be greater than wheel radius',
 'Chain stay should be greater than BB',
 'Seat stay should be greater than wheel radius',
 'Down tube must reach head tube',
 "The pedal shouldn't intersect the front wheel",
 "The crank shouldn't hit the ground when it is in its lower position",
 'RGB value should be less than 255',
 'Predicted Frame Validity']

In [6]:
ref_point

array([ 7.9141116e-01,  2.7965340e+01,  1.8633813e+02,  8.0988870e+02,
        8.4861957e+02,  2.2190498e+01,  1.8032625e+02,  2.6500977e+02,
        1.1630887e+02,  1.5000000e+00,  1.5000000e+00,  6.2000000e+02,
        8.5269995e+02,  4.0000000e+01,  2.8000000e+02,  8.3830000e+02,
        3.6800000e+02,  0.0000000e+00,  9.9736510e+01,  1.2588989e+01,
        5.0950000e+02,  3.7500000e+01,  0.0000000e+00, -5.0000000e-01])

In [7]:
ref_point

array([ 7.9141116e-01,  2.7965340e+01,  1.8633813e+02,  8.0988870e+02,
        8.4861957e+02,  2.2190498e+01,  1.8032625e+02,  2.6500977e+02,
        1.1630887e+02,  1.5000000e+00,  1.5000000e+00,  6.2000000e+02,
        8.5269995e+02,  4.0000000e+01,  2.8000000e+02,  8.3830000e+02,
        3.6800000e+02,  0.0000000e+00,  9.9736510e+01,  1.2588989e+01,
        5.0950000e+02,  3.7500000e+01,  0.0000000e+00, -5.0000000e-01])

In [8]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)
data_tens = torch.tensor(data.values, dtype=torch.float32)

evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)

ref_point = get_ref_point(evaluator, requirement_names)

assert ref_point.min() > 0, "Ref point should be greater than 0"

def calc_composite_score(data_tens, condition, evaluator = evaluator):
    # Calculate the composite score for each row in the data tensor
    eval_scores = evaluator(data_tens, condition)
    scaled_scores = eval_scores / ref_point
    

    

AssertionError: Ref point should be greater than 0